In [0]:
"""
02_quality_kpis.py

Streaming Quality KPIs.

Computes product quality KPIs from
Silver Quality Events.

Input:
    quality_events

Output:
    quality_kpis

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    sum,
    when,
)


# ============================================================
# Quality KPIs
# ============================================================

@dlt.table(
    name="quality_kpis",
    comment="Product quality KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect_or_drop(
    "valid_product_code",
    "product_code IS NOT NULL",
)

@dlt.expect_or_drop(
    "valid_test_program",
    "test_program_id IS NOT NULL",
)

@dlt.expect(
    "positive_tests_completed",
    "tests_completed > 0",
)

def quality_kpis():

    quality = (
        dlt.read_stream(
            "quality_events"
        )
    .withWatermark(
        "event_timestamp",
        "10 minutes",
    )
)

    return (

        quality

        .groupBy(

            "plant_code",

            "product_code",

            "product_name",

            "family",

            "test_program_id",

            "test_name",

        )

        .agg(

            count("*").alias(
                "tests_completed"
            ),

            sum(

                when(
                    col("result") == "PASS",
                    1,
                ).otherwise(0)

            ).alias(
                "tests_passed"
            ),

            sum(

                when(
                    col("result") == "FAIL",
                    1,
                ).otherwise(0)

            ).alias(
                "tests_failed"
            ),

        )

        .withColumn(

            "pass_rate",

            (
                col("tests_passed")
                /
                col("tests_completed")
            ) * 100

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )